In [0]:
%python
from pyspark.sql.functions import *
orders_bronze_df = spark.table("retails_project.bronze.orders_bronze")

silver_orders_df = orders_bronze_df.withColumn(
    "order_date",
    coalesce(
        try_to_date(col("order_date"), "yyyy-MM-dd"),
        try_to_date(col("order_date"), "dd-MM-yyyy"),
        try_to_date(col("order_date"), "yyyy/MM/dd"),
        try_to_date(col("order_date"), "dd/MM/yyyy")
    )
)
silver_orders_df.display()

In [0]:
%python
silver_orders_df.select("category").distinct().display()

In [0]:
%python
silver_orders_df = silver_orders_df.withColumn(
    "category",
    upper(trim(col("category")))
)


silver_orders_df = silver_orders_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

In [0]:
%python
silver_orders_df=silver_orders_df.dropDuplicates(["order_id"])
silver_orders_df.select(count("order_id"), countDistinct("order_id")).display()

In [0]:
%python
silver_orders_df=silver_orders_df.withColumn(
    "discount",
    when(col("discount").isNull(), '0').when(col("discount")=="ten", '0.1').otherwise(col("discount"))
)

silver_orders_df=silver_orders_df.withColumn(
    "discount",
    when(col("discount").like("5%"),
         regexp_replace(col("discount"), "5%", "0.05")).
    when(col("discount").like("10%"),
         regexp_replace(col("discount"), "10%", "0.1")).
    otherwise(col("discount"))
)

In [0]:
%python
silver_orders_df=silver_orders_df.withColumn(
    "discount",
    when(col("discount").like("5%"),
         regexp_replace(col("discount"), "5%", "0.05")).
    when(col("discount").like("10%"),
         regexp_replace(col("discount"), "10%", "0.1")).
    otherwise(col("discount"))
)
# MAGIC Handling negative quantity

silver_orders_df=silver_orders_df.filter(col("quantity")>=0)



silver_orders_df.display()

In [0]:
%python
from pyspark.sql.types import DecimalType

silver_orders_df=silver_orders_df.withColumn(
    "unit_price",
    col("unit_price").try_cast(DecimalType(10,2))
)
silver_orders_df.display()

In [0]:
%python
silver_orders_df=silver_orders_df.withColumn(
    "total_amount",
    col("quantity").cast("int")*col("unit_price")*(1-col("discount").cast('double'))
)
silver_orders_df = silver_orders_df.orderBy("order_id")
silver_orders_df.display()
silver_orders_df.write.format("delta").mode("overwrite").saveAsTable("retails_project.silver.orders_silver")

In [0]:
%python
customers_bronze_df=spark.table("retails_project.bronze.customers_bronze")

# COMMAND ----------
from pyspark.sql.functions import *

silver_customers_df=customers_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_time")
)
silver_customers_df.display()

In [0]:
%python
from pyspark.sql.window import Window

window_spec=Window.partitionBy('customer_id') \
    .orderBy(desc("ingest_time"))

silver_customers_df=silver_customers_df.withColumn(
    "row_num",
    row_number().over(window_spec)
)

silver_customers_df=silver_customers_df.filter("row_num = 1") \
    .drop("row_num")

silver_customers_df.display()

In [0]:
%python
silver_customers_df=silver_customers_df.withColumn(
    "status",
    upper(col("status"))
)

silver_customers_df=silver_customers_df.withColumn(
    "status",
    when(col("status").isin("ACTIVE", "INACTIVE"), col("status"))
    .otherwise("UNKNOWN")
)
silver_customers_df.display()

In [0]:
%python
silver_customers_df=silver_customers_df.withColumn(
    "loyalty_points",
    regexp_replace(col("loyalty_points"), r"[^0-9]", "")
)

silver_customers_df=silver_customers_df.withColumn(
    "loyalty_points",
    col("loyalty_points").try_cast("int")
)


In [0]:
%python
silver_customers_df=silver_customers_df.withColumn(
    "email_valid",
    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$")
)

silver_customers_df.filter(col("email_valid")=="True")


In [0]:
%python
from pyspark.sql.functions import *

silver_customers_df = silver_customers_df.withColumn(
    "dob_parsed",
    coalesce(
            try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
            try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
            try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
            try_to_date(col('date_of_birth'),'dd/MM/yyyy')
        )
)

silver_customers_df = silver_customers_df.withColumn(
    "registration_parsed",
    coalesce(
            try_to_date(col('registration_date'),'yyyy-MM-dd'),
            try_to_date(col('registration_date'),'dd-MM-yyyy'),
            try_to_date(col('registration_date'),'yyyy/MM/dd'),
            try_to_date(col('registration_date'),'dd/MM/yyyy')
        )
)

silver_customers_df.display()

In [0]:
%python
silver_customers_df=silver_customers_df.drop('date_of_birth', 'registration_date') \
    .withColumnRenamed('dob_parsed', 'date_of_birth') \
    .withColumnRenamed('registration_parsed', 'registration_date')

silver_customers_df=silver_customers_df.withColumn(
    "gender",
    upper(col("gender"))
)

silver_customers_df=silver_customers_df.withColumn(
    "gender",
    when(col("gender")=="MALE","M").when(col("gender")=="FEMALE","F").otherwise("U")
)

silver_customers_df.display()

In [0]:
%python
silver_customers_df.write.format('delta').mode('overwrite').saveAsTable('retails_project.silver.customers_silver')